### Installing Utilities and Libraries

In [ ]:
%pip install \
    databricks-sdk==0.49.0 \
    openai-agents==0.22.0 \
    mcp==2.0.0 \
    databricks-mcp==0.9.2 \
    "mlflow>=3.1"

### Restart your Python Environment

In [ ]:
dbutils.library.restartPython()

### Set up the Environment

In [ ]:
from databricks.sdk import WorkspaceClient

# Get Databricks runtime authentication
w = WorkspaceClient()

headers = w.config.authenticate()
token = headers["Authorization"].replace("Bearer ", "")
workspace_host = w.config.host.rstrip("/")

### Define your MCP Server

In [ ]:
from databricks_mcp import DatabricksMCPClient
from databricks.sdk import WorkspaceClient

mcp_server_url = "YOUR_MCP_SERVER_URL_GOES_HERE"


### Create and Execute the Agent

In [ ]:
from agents import (
    Agent,
    Runner,
    AsyncOpenAI,
    OpenAIChatCompletionsModel,
    set_tracing_disabled
)
from agents.mcp import MCPServerStreamableHttp

# Create an OpenAI-compatible client for Databricks
client = AsyncOpenAI(
    api_key=token,
    base_url=f"{workspace_host}/serving-endpoints"
)

# Configure the Databricks model
model = OpenAIChatCompletionsModel(
    model="databricks-claude-sonnet-4-5",
    openai_client=client
)

async with MCPServerStreamableHttp(
    name="Recipes-and-Courses-MCP-Server",
    params={
        "url": mcp_server_url,
        "headers": {
            "Authorization": f"Bearer {token}"
        }
    }
) as mcp_server:

    # Don't send Agents SDK traces to OpenAI
    set_tracing_disabled(True)
    
    # Create agent
    agent = Agent(
        name="Courses-Recipes-Agent",
        instructions=(
            "You are a helpful AI assistant with access to a custom "
            "MCP server containing information about courses and recipes. "
            "Use the available MCP tools whenever the user asks about "
            "courses, course ratings, UFB courses, or recipes."
        ),
        model=model,
        mcp_servers=[mcp_server]
    )

    # Run agent
    result = await Runner.run(
        agent,
        "Show me all courses with a rating of at least 4.5."
    )

    print(result.final_output)